# FA-UNIFEWS: Complete Experiment Suite v2

**Target: NeurIPS 2026** — 13 experiments, 9 datasets, 6+ methods, publication-quality figures.

## Experiment Index
| # | Experiment | Purpose | Est. Time |
|---|---|---|---|
| 1 | Main Comparison Table | 9 datasets × 6 methods | ~1h (1-seed) / ~5h (10-seed) |
| 2 | Computational Shock Demo | Prove Definition 1 visually | ~30 min |
| 3 | Edge Homophily Purification | Graph purification evidence | ~10 min |
| 4 | Multi-Backbone (GCN/GAT/SAGE/GCNII) | Backbone transferability | ~1h |
| 5 | Deep GNN (2→16 layers) | Depth robustness | ~30 min |
| 6 | Decoupled Backbone (SGC/APPNP) | Scalable backbone support | ~30 min |
| 7 | Ablation Study | Component contribution | ~30 min |
| 8 | Sparsity-Accuracy Sweep | Robustness across τ_a | ~1.5h |
| 9 | Gating α Distribution | Interpretability | ~10 min |
| 10 | Convergence Analysis | Training dynamics | ~20 min |
| 11 | Efficiency Analysis | MACs overhead | post-processing |
| 12 | t-SNE Embeddings | Qualitative visualization | ~10 min |
| 13 | Statistical Significance | p-values | post-processing |

**Run order**: 0 (Setup) → Quick Test → EXP 1 (1-seed) → EXP 2-3 → EXP 4-8 → EXP 9-12 → EXP 1 (10-seed) → EXP 13 → Figures

---
## 0. Setup

In [ ]:
# ── Detect environment ──
import os, sys, re, time, json, subprocess, csv
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Research/ConfA/Project 2')
else:
    PROJECT_ROOT = Path('/Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2')

UNIFEWS_DIR = PROJECT_ROOT / 'Unifews'
CONFIG_DIR  = UNIFEWS_DIR / 'config'
DATA_DIR    = UNIFEWS_DIR / 'data'
SAVE_DIR    = UNIFEWS_DIR / 'save'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / 'figures').mkdir(exist_ok=True)
(RESULTS_DIR / 'tables').mkdir(exist_ok=True)

os.chdir(UNIFEWS_DIR)
sys.path.insert(0, str(UNIFEWS_DIR))

# Verify datasets
datasets_on_disk = sorted([d.name for d in DATA_DIR.iterdir() 
                           if d.is_dir() and (d / 'adj.npz').exists()])
print(f'Working dir: {os.getcwd()}')
print(f'Datasets on disk: {datasets_on_disk}')
print(f'Configs available: {sorted([f.stem for f in CONFIG_DIR.glob("*.json")])}')

In [ ]:
# ── Download missing datasets ──
import torch
import scipy.sparse as sp

def download_and_convert(dataset_name, data_dir):
    """Download a PyG dataset and save in Unifews format."""
    from torch_geometric.datasets import WikipediaNetwork, WebKB, Planetoid
    save_path = data_dir / dataset_name
    if save_path.exists() and (save_path / 'adj.npz').exists():
        print(f'  {dataset_name}: exists, skip')
        return
    save_path.mkdir(parents=True, exist_ok=True)
    tmp = data_dir / 'raw'; tmp.mkdir(exist_ok=True)
    
    if dataset_name in ['chameleon', 'squirrel']:
        ds = WikipediaNetwork(root=str(tmp), name=dataset_name)
    elif dataset_name in ['cornell', 'texas', 'wisconsin']:
        ds = WebKB(root=str(tmp), name=dataset_name)
    elif dataset_name in ['citeseer', 'pubmed']:
        ds = Planetoid(root=str(tmp), name=dataset_name)
    else:
        raise ValueError(f'Unknown: {dataset_name}')
    
    g = ds[0]; n = g.num_nodes
    row, col = g.edge_index[0].numpy(), g.edge_index[1].numpy()
    r2, c2 = np.concatenate([row,col]), np.concatenate([col,row])
    adj = sp.csr_matrix((np.ones(len(r2), dtype=np.int8), (r2, c2)), shape=(n,n))
    adj.setdiag(0); adj.eliminate_zeros(); adj.data = np.ones(adj.nnz, dtype=np.int8)
    feats = g.x.numpy()
    labels = g.y.numpy().flatten()
    
    if hasattr(g, 'train_mask') and g.train_mask is not None:
        m = g.train_mask
        if m.dim() > 1: m = m[:, 0]
        idx_train = torch.where(m)[0].numpy()
        m = g.val_mask if g.val_mask.dim() == 1 else g.val_mask[:, 0]
        idx_val = torch.where(m)[0].numpy()
        m = g.test_mask if g.test_mask.dim() == 1 else g.test_mask[:, 0]
        idx_test = torch.where(m)[0].numpy()
    else:
        perm = np.random.RandomState(42).permutation(n)
        n1, n2 = int(0.5*n), int(0.75*n)
        idx_train, idx_val, idx_test = perm[:n1], perm[n1:n2], perm[n2:]
    
    sp.save_npz(str(save_path/'adj.npz'), adj.tocsc())
    np.save(str(save_path/'feats.npy'), feats)
    np.savez(str(save_path/'labels.npz'), labels=labels,
             idx_train=idx_train, idx_val=idx_val, idx_test=idx_test)
    deg = np.array(adj.sum(1)).flatten()
    sp.save_npz(str(save_path/'degree.npz'), sp.csr_matrix(deg.reshape(-1,1)))
    
    rows_, cols_ = adj.nonzero()
    h = (labels[rows_] == labels[cols_]).mean() if len(rows_) > 0 else 0
    print(f'  {dataset_name}: n={n}, m={adj.nnz}, f={feats.shape[1]}, '
          f'c={len(np.unique(labels))}, h={h:.3f}')

for ds in ['chameleon', 'cornell', 'wisconsin', 'texas', 'citeseer', 'pubmed']:
    try: download_and_convert(ds, DATA_DIR)
    except Exception as e: print(f'  {ds}: FAILED - {e}')

# Verify all datasets
print('\n=== Dataset Summary ===')
for d in sorted(DATA_DIR.iterdir()):
    if d.is_dir() and (d / 'adj.npz').exists():
        a = sp.load_npz(str(d/'adj.npz'))
        l = np.load(str(d/'labels.npz'), allow_pickle=True)
        r, c = a.nonzero()
        h = (l['labels'][r] == l['labels'][c]).mean() if len(r) > 0 else 0
        print(f'  {d.name:15s}: n={a.shape[0]:6d}, m={a.nnz:7d}, '
              f'c={len(np.unique(l["labels"])):2d}, h={h:.3f}')

In [ ]:
# ── Matplotlib setup (publication quality, no LaTeX) ──
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext

# Safe set_text to avoid recursion with non-string inputs
def _safe_set_text(self, s):
    if s is None:
        s = ''
    else:
        s = str(s)
        # Strip LaTeX commands for non-LaTeX mode
        s = re.sub(r'\\textbf\{([^}]*)\}', r'\1', s)
        s = re.sub(r'\\textit\{([^}]*)\}', r'\1', s)
        s = re.sub(r'\\%', '%', s)
    self._text = s
    self.stale = True

mtext.Text.set_text = _safe_set_text

plt.close('all')
plt.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Color-blind friendly palette
COLORS = {
    'Dense GCN': '#4477AA',   # blue
    'MLP':       '#BBBBBB',   # gray
    'Random':    '#EE6677',   # red
    'UNIFEWS':   '#228833',   # green
    'FA-static': '#CCBB44',   # yellow
    'FA-adapt':  '#AA3377',   # purple
}
METHOD_ORDER = ['Dense GCN', 'MLP', 'Random', 'UNIFEWS', 'FA-static', 'FA-adapt']

# NeurIPS column widths
COL1, COL2 = 3.25, 6.75

print('Matplotlib ready. Backend:', matplotlib.get_backend())

In [ ]:
# ── Core experiment runner ──
def run_experiment(config, algo='gcn_thr', seed=42, thr_a=0.5, thr_w=0.5,
                   fa_alpha=1.0, device=0, extra_args=None, timeout=600):
    """
    Run a single experiment via run_fb.py subprocess.
    Returns dict with acc, numel_a, numel_w, time_train, time_test, macs_test.
    """
    cmd = ['python', 'run_fb.py',
           '-f', str(seed), '-c', config, '-m', algo,
           '-a', str(thr_a), '-w', str(thr_w), '-v', str(device),
           '--fa_alpha', str(fa_alpha)]
    if extra_args:
        cmd.extend(extra_args)
    
    result = {
        'config': config, 'algo': algo, 'seed': seed,
        'thr_a': thr_a, 'thr_w': thr_w, 'fa_alpha': fa_alpha,
        'acc': None, 'numel_a': None, 'numel_w': None,
        'time_train': None, 'time_test': None, 'macs_test': None,
        'macs_train': None, 'error': None,
    }
    
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True,
                              cwd=str(UNIFEWS_DIR), timeout=timeout)
    except subprocess.TimeoutExpired:
        result['error'] = 'timeout'
        return result
    
    if proc.returncode != 0:
        result['error'] = proc.stderr[-500:] if proc.stderr else 'unknown'
        return result
    
    output = proc.stdout + proc.stderr
    for line in output.split('\n'):
        ll = line.lower()
        # [Val] best acc: X, [Test] best acc: Y
        if '[test]' in ll and 'best acc' in ll:
            m = re.search(r'best acc:\s*([0-9.]+)', line, re.I)
            if m: result['acc'] = float(m.group(1))
        # [Test]  time: X s, MACs: Y G, Num adj: Z k, Num weight: W k
        if '[test]' in ll and 'num adj' in ll:
            for key, pat in [('numel_a', r'Num adj:\s*([0-9.]+)'),
                             ('numel_w', r'Num weight:\s*([0-9.]+)'),
                             ('time_test', r'time:\s*([0-9.]+)'),
                             ('macs_test', r'MACs:\s*([0-9.]+)')]:
                m = re.search(pat, line)
                if m: result[key] = float(m.group(1))
        # [Train] time: X s
        if '[train]' in ll and 'time:' in ll:
            m = re.search(r'time:\s*([0-9.]+)', line)
            if m: result['time_train'] = float(m.group(1))
            m = re.search(r'MACs:\s*([0-9.]+)', line)
            if m: result['macs_train'] = float(m.group(1))
    
    return result


def run_batch(experiments, desc='', save_intermediate=True):
    """Run a batch of experiments with progress tracking."""
    results = []
    total = len(experiments)
    t0 = time.time()
    print(f'\n{"="*60}')
    print(f'{desc} ({total} runs)')
    print(f'{"="*60}')
    
    for i, exp in enumerate(experiments):
        label = f'{exp.get("config","?")}/{exp.get("algo","?")}'
        fa = exp.get('fa_alpha', 1.0)
        print(f'  [{i+1}/{total}] {label} fa={fa}', end=' ', flush=True)
        try:
            r = run_experiment(**exp)
            results.append(r)
            if r.get('acc'):
                print(f'→ {r["acc"]*100:.2f}%', flush=True)
            else:
                print(f'→ FAILED: {r.get("error","?")[:60]}', flush=True)
        except Exception as e:
            print(f'→ EXCEPTION: {e}', flush=True)
            results.append({**exp, 'acc': None, 'error': str(e)})
    
    elapsed = time.time() - t0
    ok = sum(1 for r in results if r.get('acc'))
    print(f'\nDone: {ok}/{total} succeeded in {elapsed:.0f}s')
    
    if save_intermediate and results:
        tag = desc.replace(' ', '_').lower()[:30]
        pd.DataFrame(results).to_csv(RESULTS_DIR / f'intermediate_{tag}.csv', index=False)
    
    return results

print('Experiment runner ready.')

In [ ]:
# ── Quick test: single run on Cora ──
print('Quick test: Cora + gcn_thr + fa_alpha=-1.0')
test_r = run_experiment('cora', 'gcn_thr', seed=42, thr_a=0.7, thr_w=0.5,
                        fa_alpha=-1.0, device=0)
print(json.dumps({k: v for k, v in test_r.items() 
                  if k not in ['stdout','stderr']}, indent=2))
assert test_r.get('acc'), f'Quick test FAILED: {test_r.get("error")}'

---
## EXP 1: Main Comparison Table (9 datasets × 6 methods)

This is the CORE result. Run single-seed first for fast feedback, then 10-seed for paper.

In [ ]:
# ── Define all datasets and methods ──
# Homophilic
DS_HOMO = ['cora', 'citeseer', 'pubmed', 'computers', 'cs']
# Heterophilic
DS_HETERO = ['chameleon', 'cornell', 'texas', 'wisconsin']
# All
DS_ALL = DS_HOMO + DS_HETERO

# Method definitions: (label, algo, fa_alpha, thr_a, thr_w)
METHODS = {
    'Dense GCN': ('gcn',     1.0, 0.0, 0.0),
    'MLP':       ('mlp',     1.0, 0.0, 0.0),
    'Random':    ('gcn_rnd', 1.0, 0.7, 0.5),
    'UNIFEWS':   ('gcn_thr', 1.0, 0.7, 0.5),
    'FA-static': ('gcn_thr', 0.5, 0.7, 0.5),
    'FA-adapt':  ('gcn_thr',-1.0, 0.7, 0.5),
}

def make_experiments(datasets, methods, seeds):
    exps = []
    for ds in datasets:
        for mname, (algo, fa, ta, tw) in methods.items():
            for s in seeds:
                exps.append(dict(config=ds, algo=algo, seed=s,
                                 thr_a=ta, thr_w=tw, fa_alpha=fa))
    return exps

# Single-seed quick pass
exps_main_1seed = make_experiments(DS_ALL, METHODS, [42])
print(f'EXP 1 (1-seed): {len(exps_main_1seed)} runs')
print(f'EXP 1 (10-seed): {len(exps_main_1seed)*10} runs')

In [ ]:
# ── Run EXP 1 single-seed (fast feedback) ──
res_main_1seed = run_batch(exps_main_1seed, desc='EXP1 Main Table (1-seed)')

In [ ]:
# ── Display EXP 1 results as table ──
def make_main_table(results, methods_map=METHODS):
    """Create a clean comparison table from results."""
    rows = []
    for r in results:
        if r.get('acc') is None: continue
        # Reverse-map to method name
        mname = 'unknown'
        for name, (algo, fa, ta, tw) in methods_map.items():
            if (r['algo'] == algo and abs(r['fa_alpha'] - fa) < 0.01
                and abs(r['thr_a'] - ta) < 0.01):
                mname = name; break
        rows.append({'Dataset': r['config'], 'Method': mname,
                     'Seed': r['seed'], 'Acc': r['acc']*100})
    df = pd.DataFrame(rows)
    if len(df) == 0:
        print('No results yet.'); return pd.DataFrame()
    pivot = df.groupby(['Dataset','Method'])['Acc'].agg(['mean','std']).reset_index()
    pivot['cell'] = pivot.apply(
        lambda x: f"{x['mean']:.2f}" if pd.isna(x['std']) or x['std']==0 
                  else f"{x['mean']:.2f}±{x['std']:.2f}", axis=1)
    table = pivot.pivot(index='Method', columns='Dataset', values='cell')
    # Reorder
    morder = [m for m in METHOD_ORDER if m in table.index]
    dorder = [d for d in DS_ALL if d in table.columns]
    table = table.loc[morder, dorder]
    return table

table1 = make_main_table(res_main_1seed)
print('\n=== EXP 1: Main Comparison (1-seed) ===')
print(table1.to_string())

---
## EXP 2: Computational Shock Demonstration

Sweep τ_a on heterophilic datasets to show UNIFEWS drops below MLP.

In [ ]:
# ── EXP 2: Sparsity sweep on heterophilic datasets ──
THR_SWEEP_SHOCK = [0.0, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]
SHOCK_DS = ['chameleon', 'cornell', 'texas', 'wisconsin']
SHOCK_METHODS = {
    'MLP':     ('mlp',     1.0),
    'UNIFEWS': ('gcn_thr', 1.0),
    'FA-adapt':('gcn_thr',-1.0),
}

exps_shock = []
# MLP only needs one τ_a (structure-free)
for ds in SHOCK_DS:
    exps_shock.append(dict(config=ds, algo='mlp', seed=42,
                           thr_a=0.0, thr_w=0.0, fa_alpha=1.0))
# GNN methods: sweep τ_a
for ds in SHOCK_DS:
    for thr in THR_SWEEP_SHOCK:
        for mname, (algo, fa) in list(SHOCK_METHODS.items())[1:]:
            exps_shock.append(dict(config=ds, algo=algo, seed=42,
                                   thr_a=thr, thr_w=0.0, fa_alpha=fa))

print(f'EXP 2: {len(exps_shock)} runs')
res_shock = run_batch(exps_shock, desc='EXP2 Computational Shock')

In [ ]:
# ── Fig 1: Computational Shock visualization ──
fig, axes = plt.subplots(1, len(SHOCK_DS), figsize=(COL2, 2.4), sharey=False)
if len(SHOCK_DS) == 1: axes = [axes]

for ax, ds in zip(axes, SHOCK_DS):
    # MLP baseline (horizontal line)
    mlp_acc = None
    for r in res_shock:
        if r['config'] == ds and r['algo'] == 'mlp' and r.get('acc'):
            mlp_acc = r['acc'] * 100
    if mlp_acc:
        ax.axhline(mlp_acc, color=COLORS['MLP'], ls='--', lw=1.5, label='MLP')
    
    # UNIFEWS and FA-adapt curves
    for mname in ['UNIFEWS', 'FA-adapt']:
        algo, fa = SHOCK_METHODS[mname]
        xs, ys = [], []
        for r in res_shock:
            if (r['config'] == ds and r['algo'] == algo 
                and abs(r['fa_alpha'] - fa) < 0.01 and r.get('acc')):
                xs.append(r['thr_a'])
                ys.append(r['acc'] * 100)
        if xs:
            order = np.argsort(xs)
            ax.plot([xs[i] for i in order], [ys[i] for i in order],
                    'o-', color=COLORS[mname], label=mname, markersize=4, lw=1.5)
    
    # Shade "shock zone" (below MLP)
    if mlp_acc:
        ax.axhspan(ax.get_ylim()[0], mlp_acc, alpha=0.08, color='red')
        ax.text(0.95, 0.05, 'Comp. Shock Zone', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=7, color='red', alpha=0.7)
    
    ax.set_title(ds.capitalize(), fontweight='bold')
    ax.set_xlabel('Threshold τ_a')
    if ax == axes[0]: ax.set_ylabel('Test Accuracy (%)')

axes[-1].legend(loc='lower left', framealpha=0.9)
fig.suptitle('Figure 1: Computational Shock — UNIFEWS vs FA-UNIFEWS',
             fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / 'figures' / 'fig1_computational_shock.pdf'))
plt.show()

---
## EXP 3: Edge Homophily Purification

In [ ]:
# ── Compute original edge homophily for all datasets ──
import scipy.sparse as sp

def compute_edge_homophily(ds_name):
    p = DATA_DIR / ds_name
    adj = sp.load_npz(str(p / 'adj.npz'))
    labels = np.load(str(p / 'labels.npz'), allow_pickle=True)['labels']
    rows, cols = adj.nonzero()
    if len(rows) == 0: return 0.0
    return (labels[rows] == labels[cols]).mean()

print('=== Original Edge Homophily ===')
homo_original = {}
for ds in DS_ALL:
    try:
        h = compute_edge_homophily(ds)
        homo_original[ds] = h
        print(f'  {ds:15s}: h = {h:.4f}')
    except Exception as e:
        print(f'  {ds:15s}: ERROR - {e}')

In [ ]:
# ── Extract pruned-graph homophily via model internals ──
# This requires running the model and capturing the pruned edge set
HOMO_SCRIPT = '''
import sys, json, os, torch, numpy as np, scipy.sparse as sp
sys.path.insert(0, "{unifews_dir}")
os.chdir("{unifews_dir}")

from utils.loader import load_edgelist
from archs import identity_n_norm
import archs.models as models
from utils.logger import prepare_opt
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('-f','--seed',type=int,default=42)
parser.add_argument('-v','--dev',type=int,default=-1)
parser.add_argument('-c','--config',type=str,default='{dataset}')
parser.add_argument('-m','--algo',type=str,default='gcn_thr')
parser.add_argument('-a','--thr_a',type=float,default={thr_a})
parser.add_argument('-w','--thr_w',type=float,default=0.5)
parser.add_argument('-l','--layer',type=int,default=None)
parser.add_argument('-n','--suffix',type=str,default='')
parser.add_argument('--fa_alpha',type=float,default={fa_alpha})
args = prepare_opt(parser)

import random
random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
device = torch.device('cpu')

adj, feat, labels, idx, nfeat, nclass = load_edgelist(
    datastr=args.data, datapath=args.path,
    inductive=args.inductive, multil=args.multil, seed=args.seed)

model = models.GNNThr(nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden,
    nclass=nclass, thr_a=args.thr_a, thr_w=args.thr_w,
    fa_alpha=args.fa_alpha, dropout=args.dropout, layer=args.algo)
model.reset_parameters()
model.kwargs['diag'] = None
adj['train'] = identity_n_norm(adj['train'], edge_weight=None,
    num_nodes=feat['train'].shape[0], rnorm=model.kwargs['rnorm'], diag=None)

# Train briefly (50 epochs) to get meaningful pruning
import torch.optim as optim, torch.nn as nn
optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
loss_fn = nn.CrossEntropyLoss()
for ep in range(50):
    model.train()
    if ep < 25: model.set_scheme('pruneall','pruneall')
    else: model.set_scheme('pruneall','pruneinc')
    optimizer.zero_grad()
    out = model(feat['train'], adj['train'], node_lock=torch.Tensor([]))[idx['train']]
    loss = loss_fn(out, labels['train'])
    loss.backward(); optimizer.step()

# Evaluate: capture pruned edges
model.eval()
model.set_scheme('keep','keep')
with torch.no_grad():
    out = model(feat['test'], adj['test'] if 'test' in adj else adj['train'],
                node_lock=idx['test'], verbose=True)

# Get pruned edge index from first layer
layer0 = model.convs[0]
if hasattr(layer0, 'logger_a'):
    # Compute homophily of retained edges
    # The adj used is the test adj
    adj_test = adj['test'] if 'test' in adj else adj['train']
    if isinstance(adj_test, tuple):
        edge_index = adj_test[0]  # (2, E)
    else:
        edge_index = adj_test
    
    # Get all labels
    lbl_path = os.path.join(args.path, args.data, 'labels.npz')
    lbl_data = np.load(lbl_path, allow_pickle=True)
    all_labels = lbl_data['labels']
    
    numel_before = layer0.logger_a.numel_before
    numel_after = layer0.logger_a.numel_after
    
    result = {{
        'dataset': '{dataset}',
        'fa_alpha': {fa_alpha},
        'numel_before': int(numel_before),
        'numel_after': int(numel_after),
        'retention': numel_after / max(numel_before, 1),
    }}
    print('HOMO_RESULT:' + json.dumps(result))
else:
    print('HOMO_RESULT:' + json.dumps({{'error': 'no logger_a'}}))
'''

# Run for UNIFEWS and FA-adapt on each dataset
homo_results = {}
for ds in DS_ALL:
    homo_results[ds] = {'original': homo_original.get(ds, None)}
    for label, fa_alpha in [('UNIFEWS', 1.0), ('FA-adapt', -1.0)]:
        script = HOMO_SCRIPT.format(
            unifews_dir=str(UNIFEWS_DIR), dataset=ds,
            thr_a=0.7, fa_alpha=fa_alpha)
        try:
            proc = subprocess.run(
                ['python', '-c', script], capture_output=True, text=True,
                cwd=str(UNIFEWS_DIR), timeout=120)
            for line in (proc.stdout + proc.stderr).split('\n'):
                if 'HOMO_RESULT:' in line:
                    data = json.loads(line.split('HOMO_RESULT:')[1])
                    homo_results[ds][label] = data
                    print(f'  {ds}/{label}: retention={data.get("retention","?")*100:.1f}%')
        except Exception as e:
            print(f'  {ds}/{label}: {e}')

print('\nHomophily extraction done.')

In [ ]:
# ── Fig 2: Edge Homophily Purification ──
plot_ds = [ds for ds in DS_ALL if ds in homo_original]
fig, ax = plt.subplots(figsize=(COL2, 2.8))

x_pos = np.arange(len(plot_ds))
w = 0.25
bars_orig = [homo_original.get(ds, 0) for ds in plot_ds]

ax.bar(x_pos - w, bars_orig, w, color='#4477AA', label='Original', edgecolor='white', lw=0.5)
# If we have pruned homophily data, add those bars
# (placeholder - the actual pruned homophily requires deeper model inspection)

ax.set_xticks(x_pos)
ax.set_xticklabels([d.capitalize() for d in plot_ds], rotation=30, ha='right')
ax.set_ylabel('Edge Homophily Ratio')
ax.axhline(0.5, color='gray', ls=':', lw=0.8, alpha=0.5)
ax.legend()
ax.set_title('Figure 2: Edge Homophily — Original vs Pruned', fontweight='bold')
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / 'figures' / 'fig2_homophily_purification.pdf'))
plt.show()

---
## EXP 4: Multi-Backbone Transferability (GCN / GAT / SAGE / GCNII)

In [ ]:
# ── EXP 4: Multi-backbone comparison ──
BACKBONES = {
    'GCN':   'gcn_thr',
    'GAT':   'gat_thr',
    'SAGE':  'gsage_thr',
    # 'GCNII': 'gcn2_thr',  # Uncomment if GCNII config works
}
BB_DS = ['cora', 'computers', 'chameleon', 'cs']
BB_MODES = {'UNIFEWS': 1.0, 'FA-static': 0.5, 'FA-adapt': -1.0}

exps_backbone = []
for ds in BB_DS:
    for bb_name, bb_algo in BACKBONES.items():
        for mode_name, fa in BB_MODES.items():
            exps_backbone.append(dict(
                config=ds, algo=bb_algo, seed=42,
                thr_a=0.7, thr_w=0.5, fa_alpha=fa))

print(f'EXP 4: {len(exps_backbone)} runs')
res_backbone = run_batch(exps_backbone, desc='EXP4 Multi-Backbone')

In [ ]:
# ── Fig 3: Multi-backbone grouped bar chart ──
bb_names_inv = {v: k for k, v in BACKBONES.items()}

fig, axes = plt.subplots(1, len(BB_DS), figsize=(COL2, 2.8), sharey=False)
if len(BB_DS) == 1: axes = [axes]

mode_colors = {'UNIFEWS': '#228833', 'FA-static': '#CCBB44', 'FA-adapt': '#AA3377'}

for ax, ds in zip(axes, BB_DS):
    bb_labels = list(BACKBONES.keys())
    x = np.arange(len(bb_labels))
    n_modes = len(BB_MODES)
    w = 0.8 / n_modes
    
    for j, (mode_name, fa) in enumerate(BB_MODES.items()):
        vals = []
        for bb_name, bb_algo in BACKBONES.items():
            acc = None
            for r in res_backbone:
                if (r['config'] == ds and r['algo'] == bb_algo
                    and abs(r['fa_alpha'] - fa) < 0.01 and r.get('acc')):
                    acc = r['acc'] * 100
            vals.append(acc if acc else 0)
        ax.bar(x + j*w - (n_modes-1)*w/2, vals, w,
               color=mode_colors[mode_name], label=mode_name if ds==BB_DS[0] else '',
               edgecolor='white', lw=0.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels(bb_labels)
    ax.set_title(ds.capitalize(), fontweight='bold')
    if ax == axes[0]: ax.set_ylabel('Test Accuracy (%)')

axes[0].legend(loc='lower left', framealpha=0.9)
fig.suptitle('Figure 3: Backbone Transferability', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / 'figures' / 'fig3_multi_backbone.pdf'))
plt.show()

---
## EXP 5: Deep GNN (2→16 layers)

In [ ]:
# ── EXP 5: Depth experiment ──
LAYERS = [2, 4, 8, 16]
DEEP_DS = ['cora', 'chameleon']
DEEP_MODES = {'UNIFEWS': 1.0, 'FA-static': 0.5, 'FA-adapt': -1.0}

exps_deep = []
for ds in DEEP_DS:
    for L in LAYERS:
        for mode_name, fa in DEEP_MODES.items():
            exps_deep.append(dict(
                config=ds, algo='gcn_thr', seed=42,
                thr_a=0.7, thr_w=0.5, fa_alpha=fa,
                extra_args=['-l', str(L)]))

print(f'EXP 5: {len(exps_deep)} runs')
res_deep = run_batch(exps_deep, desc='EXP5 Deep GNN')

In [ ]:
# ── Fig 4: Accuracy vs Depth ──
fig, axes = plt.subplots(1, len(DEEP_DS), figsize=(COL2, 2.6), sharey=False)

for ax, ds in zip(axes, DEEP_DS):
    for mode_name, fa in DEEP_MODES.items():
        xs, ys = [], []
        for r in res_deep:
            if (r['config'] == ds and abs(r['fa_alpha'] - fa) < 0.01 and r.get('acc')):
                # Extract layer count from extra_args
                ea = r.get('extra_args', [])
                L = int(ea[ea.index('-l')+1]) if '-l' in ea else 2
                xs.append(L)
                ys.append(r['acc'] * 100)
        if xs:
            order = np.argsort(xs)
            color = {'UNIFEWS':'#228833','FA-static':'#CCBB44','FA-adapt':'#AA3377'}[mode_name]
            ax.plot([xs[i] for i in order], [ys[i] for i in order],
                    'o-', color=color, label=mode_name, markersize=5, lw=1.5)
    
    ax.set_xlabel('Number of Layers')
    ax.set_xticks(LAYERS)
    ax.set_title(ds.capitalize(), fontweight='bold')
    if ax == axes[0]: ax.set_ylabel('Test Accuracy (%)')

axes[-1].legend()
fig.suptitle('Figure 4: Accuracy vs GNN Depth', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / 'figures' / 'fig4_deep_gnn.pdf'))
plt.show()

---
## EXP 6: Decoupled Backbone (SGC / APPNP)

In [ ]:
# ── EXP 6: SGC and APPNP table comparison ──
# Uses existing *_sgc_table*.json and *_appnp_table*.json configs
DECOUPLED_DS = ['cora', 'computers', 'cs']
DECOUPLED_ALGOS = [
    ('SGC (dense)',     'sgc',      '{ds}_sgc_table'),
    ('SGC + UNIFEWS',  'sgc_thr',  '{ds}_sgc_table_thr'),
    ('APPNP (dense)',  'appnp',    '{ds}_appnp_table'),
    ('APPNP + UNIFEWS','appnp_thr','{ds}_appnp_table_thr'),
]

exps_decoupled = []
for ds in DECOUPLED_DS:
    for label, algo, cfg_tmpl in DECOUPLED_ALGOS:
        cfg = cfg_tmpl.format(ds=ds)
        cfg_path = CONFIG_DIR / f'{cfg}.json'
        if cfg_path.exists():
            exps_decoupled.append(dict(
                config=cfg, algo=algo, seed=42,
                thr_a=None, thr_w=None, fa_alpha=1.0))  # Uses config defaults
        else:
            print(f'  Config missing: {cfg}.json')

print(f'EXP 6: {len(exps_decoupled)} runs')
# Note: thr_a/thr_w=None means use config defaults
# We need to handle this in run_experiment
res_decoupled = []
for exp in exps_decoupled:
    # Read thr_a/thr_w from config file
    cfg_name = exp['config']
    with open(CONFIG_DIR / f'{cfg_name}.json') as f:
        cfg_data = json.load(f)
    exp_fixed = {**exp, 'thr_a': cfg_data.get('thr_a', 0.0),
                 'thr_w': cfg_data.get('thr_w', 0.0)}
    r = run_experiment(**exp_fixed)
    res_decoupled.append(r)
    acc_s = f'{r["acc"]*100:.2f}%' if r.get('acc') else 'FAIL'
    print(f'  {cfg_name}/{exp["algo"]}: {acc_s}')

---
## EXP 7: Ablation Study

In [ ]:
# ── EXP 7: Ablation ──
ABL_DS = ['cora', 'chameleon', 'computers']
ALPHA_SWEEP = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

exps_ablation = []
for ds in ABL_DS:
    # Full adaptive (Mode 3)
    exps_ablation.append(dict(config=ds, algo='gcn_thr', seed=42,
                              thr_a=0.7, thr_w=0.5, fa_alpha=-1.0))
    # Static alpha sweep (Mode 2)
    for alpha in ALPHA_SWEEP:
        exps_ablation.append(dict(config=ds, algo='gcn_thr', seed=42,
                                  thr_a=0.7, thr_w=0.5, fa_alpha=alpha))

print(f'EXP 7: {len(exps_ablation)} runs')
res_ablation = run_batch(exps_ablation, desc='EXP7 Ablation')

In [ ]:
# ── Fig 5: Ablation — α sensitivity ──
fig, axes = plt.subplots(1, len(ABL_DS), figsize=(COL2, 2.6), sharey=False)

for ax, ds in zip(axes, ABL_DS):
    # Static alpha curve
    alphas_x, alphas_y = [], []
    adapt_acc = None
    for r in res_ablation:
        if r['config'] != ds or not r.get('acc'): continue
        if r['fa_alpha'] >= 0:
            alphas_x.append(r['fa_alpha'])
            alphas_y.append(r['acc'] * 100)
        elif r['fa_alpha'] < 0:
            adapt_acc = r['acc'] * 100
    
    if alphas_x:
        order = np.argsort(alphas_x)
        ax.plot([alphas_x[i] for i in order], [alphas_y[i] for i in order],
                'o-', color='#228833', label='Static (Mode 2)', markersize=5, lw=1.5)
    
    if adapt_acc:
        ax.axhline(adapt_acc, color='#AA3377', ls='--', lw=1.5, label='Adaptive (Mode 3)')
    
    ax.set_xlabel('Static α value')
    ax.set_title(ds.capitalize(), fontweight='bold')
    if ax == axes[0]: ax.set_ylabel('Test Accuracy (%)')

axes[-1].legend()
fig.suptitle('Figure 5: Ablation — Sensitivity to Static α', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / 'figures' / 'fig5_alpha_ablation.pdf'))
plt.show()

---
## EXP 8: Sparsity-Accuracy Trade-off Curves

In [ ]:
# ── EXP 8: Sparsity sweep ──
THR_SWEEP = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]
SPAR_DS = ['cora', 'computers', 'chameleon']
SPAR_METHODS = {
    'UNIFEWS':  1.0,
    'FA-static': 0.5,
    'FA-adapt': -1.0,
}

exps_sparsity = []
for ds in SPAR_DS:
    for thr in THR_SWEEP:
        for mname, fa in SPAR_METHODS.items():
            exps_sparsity.append(dict(
                config=ds, algo='gcn_thr', seed=42,
                thr_a=thr, thr_w=0.0, fa_alpha=fa))

print(f'EXP 8: {len(exps_sparsity)} runs')
res_sparsity = run_batch(exps_sparsity, desc='EXP8 Sparsity Sweep')

In [ ]:
# ── Fig 6: Sparsity vs Accuracy ──
fig, axes = plt.subplots(1, len(SPAR_DS), figsize=(COL2, 2.6), sharey=False)

for ax, ds in zip(axes, SPAR_DS):
    for mname, fa in SPAR_METHODS.items():
        xs, ys = [], []
        for r in res_sparsity:
            if (r['config'] == ds and abs(r['fa_alpha'] - fa) < 0.01 and r.get('acc')):
                # Use numel_a as proxy for sparsity
                xs.append(r['thr_a'])
                ys.append(r['acc'] * 100)
        if xs:
            order = np.argsort(xs)
            color = {'UNIFEWS':'#228833','FA-static':'#CCBB44','FA-adapt':'#AA3377'}[mname]
            ax.plot([xs[i] for i in order], [ys[i] for i in order],
                    'o-', color=color, label=mname, markersize=4, lw=1.5)
    
    ax.set_xlabel('Threshold τ_a')
    ax.set_title(ds.capitalize(), fontweight='bold')
    if ax == axes[0]: ax.set_ylabel('Test Accuracy (%)')

axes[-1].legend()
fig.suptitle('Figure 6: Sparsity-Accuracy Trade-off', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / 'figures' / 'fig6_sparsity_accuracy.pdf'))
plt.show()

---
## EXP 9: Gating α_i Distribution (Adaptive Mode)

In [ ]:
# ── EXP 9: Extract gate values from adaptive model ──
GATE_SCRIPT = '''
import sys, json, os, torch, numpy as np
sys.path.insert(0, "{unifews_dir}")
os.chdir("{unifews_dir}")

from utils.loader import load_edgelist
from archs import identity_n_norm
import archs.models as models
from utils.logger import prepare_opt
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('-f','--seed',type=int,default=42)
parser.add_argument('-v','--dev',type=int,default=-1)
parser.add_argument('-c','--config',type=str,default='{dataset}')
parser.add_argument('-m','--algo',type=str,default='gcn_thr')
parser.add_argument('-a','--thr_a',type=float,default=0.7)
parser.add_argument('-w','--thr_w',type=float,default=0.5)
parser.add_argument('-l','--layer',type=int,default=None)
parser.add_argument('-n','--suffix',type=str,default='')
parser.add_argument('--fa_alpha',type=float,default=-1.0)
args = prepare_opt(parser)

import random
random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
device = torch.device('cpu')

adj, feat, labels, idx, nfeat, nclass = load_edgelist(
    datastr=args.data, datapath=args.path,
    inductive=args.inductive, multil=args.multil, seed=args.seed)

model = models.GNNThr(nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden,
    nclass=nclass, thr_a=args.thr_a, thr_w=args.thr_w,
    fa_alpha=args.fa_alpha, dropout=args.dropout, layer=args.algo)
model.reset_parameters()
model.kwargs['diag'] = None
adj['train'] = identity_n_norm(adj['train'], edge_weight=None,
    num_nodes=feat['train'].shape[0], rnorm=model.kwargs['rnorm'], diag=None)

# Train
import torch.optim as optim, torch.nn as nn
optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
loss_fn = nn.CrossEntropyLoss()
for ep in range(args.epochs):
    model.train()
    if ep < args.epochs//2: model.set_scheme('pruneall','pruneall')
    else: model.set_scheme('pruneall','pruneinc')
    optimizer.zero_grad()
    out = model(feat['train'], adj['train'], node_lock=torch.Tensor([]))[idx['train']]
    loss = loss_fn(out, labels['train'])
    loss.backward(); optimizer.step()

# Extract gate values
model.eval()
with torch.no_grad():
    alphas = []
    for conv in model.convs:
        if hasattr(conv, 'alpha_mlp') and conv.alpha_mlp is not None:
            alpha_vals = torch.sigmoid(conv.alpha_mlp(feat['train'])).squeeze().numpy()
            alphas.append(alpha_vals.tolist())

if alphas:
    print('GATE_RESULT:' + json.dumps({{
        'dataset': '{dataset}',
        'n_layers': len(alphas),
        'alpha_layer0': alphas[0],
        'mean': float(np.mean(alphas[0])),
        'std': float(np.std(alphas[0])),
    }}))
else:
    print('GATE_RESULT:' + json.dumps({{'error': 'no alpha_mlp found'}}))
'''

GATE_DS = ['cora', 'chameleon', 'computers']
gate_data = {}

for ds in GATE_DS:
    script = GATE_SCRIPT.format(unifews_dir=str(UNIFEWS_DIR), dataset=ds)
    print(f'  Extracting gates for {ds}...', end=' ', flush=True)
    try:
        proc = subprocess.run(['python', '-c', script],
                              capture_output=True, text=True,
                              cwd=str(UNIFEWS_DIR), timeout=300)
        for line in (proc.stdout + proc.stderr).split('\n'):
            if 'GATE_RESULT:' in line:
                data = json.loads(line.split('GATE_RESULT:')[1])
                if 'error' not in data:
                    gate_data[ds] = data
                    print(f'mean α = {data["mean"]:.3f} ± {data["std"]:.3f}')
                else:
                    print(f'ERROR: {data["error"]}')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\nGate data collected for: {list(gate_data.keys())}')

In [ ]:
# ── Fig 7: Gate α_i distribution ──
if gate_data:
    n_ds = len(gate_data)
    fig, axes = plt.subplots(1, n_ds, figsize=(COL2, 2.2), sharey=True)
    if n_ds == 1: axes = [axes]
    
    for ax, (ds, data) in zip(axes, gate_data.items()):
        vals = np.array(data['alpha_layer0'])
        ax.hist(vals, bins=50, density=True, color='#AA3377', alpha=0.7, edgecolor='white', lw=0.5)
        ax.axvline(data['mean'], color='black', ls='--', lw=1, label=f'mean={data["mean"]:.2f}')
        ax.set_xlabel('α_i (gate value)')
        ax.set_title(ds.capitalize(), fontweight='bold')
        if ax == axes[0]: ax.set_ylabel('Density')
        ax.legend(fontsize=7)
        ax.set_xlim(0, 1)
    
    fig.suptitle('Figure 7: Learned Gate α_i Distribution (Adaptive Mode)',
                 fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(str(RESULTS_DIR / 'figures' / 'fig7_gate_distribution.pdf'))
    plt.show()
else:
    print('No gate data available.')

---
## EXP 10: Convergence Analysis

In [ ]:
# ── EXP 10: Training convergence with epoch-level logging ──
CONV_SCRIPT = '''
import sys, json, os, torch, numpy as np
sys.path.insert(0, "{unifews_dir}")
os.chdir("{unifews_dir}")

from utils.loader import load_edgelist
from archs import identity_n_norm
import archs.models as models
from utils.logger import prepare_opt
import utils.metric as metric
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('-f','--seed',type=int,default=42)
parser.add_argument('-v','--dev',type=int,default=-1)
parser.add_argument('-c','--config',type=str,default='{dataset}')
parser.add_argument('-m','--algo',type=str,default='gcn_thr')
parser.add_argument('-a','--thr_a',type=float,default=0.7)
parser.add_argument('-w','--thr_w',type=float,default=0.5)
parser.add_argument('-l','--layer',type=int,default=None)
parser.add_argument('-n','--suffix',type=str,default='')
parser.add_argument('--fa_alpha',type=float,default={fa_alpha})
args = prepare_opt(parser)

import random, torch.optim as optim, torch.nn as nn
random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
device = torch.device('cpu')

adj, feat, labels, idx, nfeat, nclass = load_edgelist(
    datastr=args.data, datapath=args.path,
    inductive=args.inductive, multil=args.multil, seed=args.seed)

if 'sgc' in args.algo or 'appnp' in args.algo:
    model = models.MLP(nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden,
        nclass=nclass, dropout=args.dropout, thr_w=args.thr_w, layer=args.algo)
elif args.algo == 'mlp':
    model = models.MLP(nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden,
        nclass=nclass, thr_w=args.thr_w, dropout=args.dropout, layer='mlp')
else:
    model = models.GNNThr(nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden,
        nclass=nclass, thr_a=args.thr_a, thr_w=args.thr_w,
        fa_alpha=args.fa_alpha, dropout=args.dropout, layer=args.algo)
model.reset_parameters()
model.kwargs['diag'] = None
adj['train'] = identity_n_norm(adj['train'], edge_weight=None,
    num_nodes=feat['train'].shape[0], rnorm=model.kwargs['rnorm'], diag=None)

optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
loss_fn = nn.CrossEntropyLoss()
calc = metric.F1Calculator(nclass)

history = []
for ep in range(1, args.epochs+1):
    model.train()
    if ep <= args.epochs//2: model.set_scheme('pruneall','pruneall')
    else: model.set_scheme('pruneall','pruneinc')
    optimizer.zero_grad()
    out = model(feat['train'], adj['train'], node_lock=torch.Tensor([]))
    loss = loss_fn(out[idx['train']], labels['train'])
    loss.backward(); optimizer.step()
    
    # Eval every 5 epochs
    if ep % 5 == 0 or ep == 1:
        model.eval()
        model.set_scheme('keep','keep')
        with torch.no_grad():
            out_v = model(feat['train'], adj['train'], node_lock=idx['val'])
            pred = out_v[idx['val']].argmax(dim=1).cpu()
            acc_v = (pred == labels['val'].cpu()).float().mean().item()
        history.append({{'epoch': ep, 'loss': loss.item(), 'val_acc': acc_v}})

print('CONV_RESULT:' + json.dumps({{
    'dataset': '{dataset}', 'fa_alpha': {fa_alpha},
    'history': history
}}))
'''

CONV_DS = ['cora', 'chameleon']
CONV_MODES = {'UNIFEWS': 1.0, 'FA-static': 0.5, 'FA-adapt': -1.0}
conv_data = defaultdict(dict)

for ds in CONV_DS:
    for mode_name, fa in CONV_MODES.items():
        script = CONV_SCRIPT.format(unifews_dir=str(UNIFEWS_DIR),
                                    dataset=ds, fa_alpha=fa)
        print(f'  {ds}/{mode_name}...', end=' ', flush=True)
        try:
            proc = subprocess.run(['python', '-c', script],
                                  capture_output=True, text=True,
                                  cwd=str(UNIFEWS_DIR), timeout=300)
            for line in (proc.stdout + proc.stderr).split('\n'):
                if 'CONV_RESULT:' in line:
                    data = json.loads(line.split('CONV_RESULT:')[1])
                    conv_data[ds][mode_name] = data['history']
                    final = data['history'][-1]['val_acc']*100 if data['history'] else 0
                    print(f'final val acc: {final:.2f}%')
        except Exception as e:
            print(f'FAILED: {e}')

print(f'\nConvergence data for: {dict(conv_data)}')

In [ ]:
# ── Fig 8: Convergence curves ──
if conv_data:
    fig, axes = plt.subplots(1, len(CONV_DS), figsize=(COL2, 2.6))
    if len(CONV_DS) == 1: axes = [axes]
    
    mode_colors = {'UNIFEWS':'#228833','FA-static':'#CCBB44','FA-adapt':'#AA3377'}
    
    for ax, ds in zip(axes, CONV_DS):
        for mode_name in CONV_MODES:
            if mode_name in conv_data.get(ds, {}):
                hist = conv_data[ds][mode_name]
                epochs = [h['epoch'] for h in hist]
                accs = [h['val_acc']*100 for h in hist]
                ax.plot(epochs, accs, color=mode_colors[mode_name],
                        label=mode_name, lw=1.5)
        
        ax.set_xlabel('Epoch')
        ax.set_title(ds.capitalize(), fontweight='bold')
        if ax == axes[0]: ax.set_ylabel('Val Accuracy (%)')
    
    axes[-1].legend()
    fig.suptitle('Figure 8: Training Convergence', fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(str(RESULTS_DIR / 'figures' / 'fig8_convergence.pdf'))
    plt.show()
else:
    print('No convergence data.')

---
## EXP 11: Efficiency Analysis

In [ ]:
# ── Fig 9: Efficiency — Accuracy vs MACs ──
# Use results from EXP 1
fig, ax = plt.subplots(figsize=(COL1+1, 3))

method_markers = {'Dense GCN':'s', 'MLP':'d', 'Random':'^',
                  'UNIFEWS':'o', 'FA-static':'v', 'FA-adapt':'*'}

for r in res_main_1seed:
    if not r.get('acc') or not r.get('macs_test'): continue
    # Determine method name
    mname = 'unknown'
    for name, (algo, fa, ta, tw) in METHODS.items():
        if (r['algo'] == algo and abs(r['fa_alpha'] - fa) < 0.01
            and abs(r['thr_a'] - ta) < 0.01):
            mname = name; break
    
    ax.scatter(r['macs_test'], r['acc']*100,
              color=COLORS.get(mname, 'gray'),
              marker=method_markers.get(mname, 'o'),
              s=60, alpha=0.8, edgecolors='black', lw=0.3,
              label=mname)

# Deduplicate legend
handles, labels_leg = ax.get_legend_handles_labels()
unique = dict(zip(labels_leg, handles))
ax.legend(unique.values(), unique.keys(), loc='lower right', fontsize=7)

ax.set_xlabel('Test MACs (G)')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Figure 9: Accuracy vs Computational Cost', fontweight='bold')
plt.tight_layout()
fig.savefig(str(RESULTS_DIR / 'figures' / 'fig9_efficiency.pdf'))
plt.show()

---
## EXP 12: t-SNE Embedding Visualization

In [ ]:
# ── EXP 12: Extract embeddings ──
EMBED_SCRIPT = '''
import sys, json, os, torch, numpy as np
sys.path.insert(0, "{unifews_dir}")
os.chdir("{unifews_dir}")

from utils.loader import load_edgelist
from archs import identity_n_norm
import archs.models as models
from utils.logger import prepare_opt
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('-f','--seed',type=int,default=42)
parser.add_argument('-v','--dev',type=int,default=-1)
parser.add_argument('-c','--config',type=str,default='{dataset}')
parser.add_argument('-m','--algo',type=str,default='gcn_thr')
parser.add_argument('-a','--thr_a',type=float,default={thr_a})
parser.add_argument('-w','--thr_w',type=float,default={thr_w})
parser.add_argument('-l','--layer',type=int,default=None)
parser.add_argument('-n','--suffix',type=str,default='')
parser.add_argument('--fa_alpha',type=float,default={fa_alpha})
args = prepare_opt(parser)

import random, torch.optim as optim, torch.nn as nn
random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
device = torch.device('cpu')

adj, feat, labels, idx, nfeat, nclass = load_edgelist(
    datastr=args.data, datapath=args.path,
    inductive=args.inductive, multil=args.multil, seed=args.seed)

if args.algo == 'mlp':
    model = models.MLP(nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden,
        nclass=nclass, thr_w=args.thr_w, dropout=args.dropout, layer='mlp')
else:
    model = models.GNNThr(nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden,
        nclass=nclass, thr_a=args.thr_a, thr_w=args.thr_w,
        fa_alpha=args.fa_alpha, dropout=args.dropout, layer=args.algo)
model.reset_parameters()
model.kwargs['diag'] = None
adj['train'] = identity_n_norm(adj['train'], edge_weight=None,
    num_nodes=feat['train'].shape[0], rnorm=model.kwargs['rnorm'], diag=None)

optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
loss_fn = nn.CrossEntropyLoss()
for ep in range(args.epochs):
    model.train()
    if ep < args.epochs//2: model.set_scheme('pruneall','pruneall')
    else: model.set_scheme('pruneall','pruneinc')
    optimizer.zero_grad()
    out = model(feat['train'], adj['train'], node_lock=torch.Tensor([]))[idx['train']]
    loss = loss_fn(out, labels['train'])
    loss.backward(); optimizer.step()

# Extract final embeddings (penultimate layer output)
model.eval()
model.set_scheme('keep','keep')
embeddings = []
hook_handle = None
def hook_fn(module, input, output):
    embeddings.append(output.detach().cpu().numpy())

# Hook the last conv layer
if hasattr(model, 'convs') and len(model.convs) > 0:
    hook_handle = model.convs[-1].register_forward_hook(hook_fn)
elif hasattr(model, 'lins') and len(model.lins) > 0:
    hook_handle = model.lins[-1].register_forward_hook(hook_fn)

with torch.no_grad():
    model(feat['test'], adj['test'] if 'test' in adj else adj['train'],
          node_lock=idx['test'])

if hook_handle: hook_handle.remove()

lbl_all = np.load(os.path.join(args.path, args.data, 'labels.npz'), allow_pickle=True)['labels']

if embeddings:
    emb = embeddings[0]
    # Subsample if too many nodes
    n = emb.shape[0]
    if n > 2000:
        idx_sub = np.random.choice(n, 2000, replace=False)
    else:
        idx_sub = np.arange(n)
    print('EMBED_RESULT:' + json.dumps({{
        'dataset': '{dataset}',
        'fa_alpha': {fa_alpha},
        'embeddings': emb[idx_sub].tolist(),
        'labels': lbl_all[idx_sub].tolist(),
    }}))
else:
    print('EMBED_RESULT:' + json.dumps({{'error': 'no embeddings captured'}}))
'''

EMBED_DS = 'cora'
EMBED_METHODS = [
    ('Dense GCN', 'gcn', 1.0, 0.0, 0.0),
    ('UNIFEWS',   'gcn_thr', 1.0, 0.7, 0.5),
    ('FA-adapt',  'gcn_thr', -1.0, 0.7, 0.5),
]

embed_data = {}
for mname, algo, fa, ta, tw in EMBED_METHODS:
    script = EMBED_SCRIPT.format(unifews_dir=str(UNIFEWS_DIR),
                                 dataset=EMBED_DS, thr_a=ta, thr_w=tw, fa_alpha=fa)
    print(f'  Extracting embeddings: {mname}...', end=' ', flush=True)
    try:
        proc = subprocess.run(['python', '-c', script],
                              capture_output=True, text=True,
                              cwd=str(UNIFEWS_DIR), timeout=300)
        for line in (proc.stdout + proc.stderr).split('\n'):
            if 'EMBED_RESULT:' in line:
                data = json.loads(line.split('EMBED_RESULT:')[1])
                if 'error' not in data:
                    embed_data[mname] = data
                    print(f'shape={np.array(data["embeddings"]).shape}')
                else:
                    print(f'ERROR: {data["error"]}')
    except Exception as e:
        print(f'FAILED: {e}')

In [ ]:
# ── Fig 10: t-SNE embeddings ──
from sklearn.manifold import TSNE

if embed_data:
    n_methods = len(embed_data)
    fig, axes = plt.subplots(1, n_methods, figsize=(COL2, 2.5))
    if n_methods == 1: axes = [axes]
    
    for ax, (mname, data) in zip(axes, embed_data.items()):
        emb = np.array(data['embeddings'])
        lbls = np.array(data['labels'])
        
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        coords = tsne.fit_transform(emb)
        
        for c in np.unique(lbls):
            mask = lbls == c
            ax.scatter(coords[mask, 0], coords[mask, 1],
                      s=5, alpha=0.6, label=f'Class {c}')
        
        ax.set_title(mname, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
    
    axes[-1].legend(markerscale=3, fontsize=6, loc='upper right')
    fig.suptitle(f'Figure 10: t-SNE Embeddings on {EMBED_DS.capitalize()}',
                 fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(str(RESULTS_DIR / 'figures' / 'fig10_tsne.pdf'))
    plt.show()
else:
    print('No embedding data.')

---
## EXP 1 (Full): 10-Seed Runs for Paper

**WARNING**: This takes ~5 hours on T4. Run overnight.

In [ ]:
# ── EXP 1 full: 10-seed ──
SEEDS_10 = list(range(1, 11))
exps_main_10seed = make_experiments(DS_ALL, METHODS, SEEDS_10)
print(f'EXP 1 (10-seed): {len(exps_main_10seed)} runs')
print('Estimated time: 4-6 hours on T4. Run at your own risk.')

In [ ]:
# ── Run 10-seed (uncomment to execute) ──
# res_main_10seed = run_batch(exps_main_10seed, desc='EXP1 Main Table (10-seed)')
# pd.DataFrame(res_main_10seed).to_csv(RESULTS_DIR / 'main_10seed_results.csv', index=False)
# print('Saved to results/main_10seed_results.csv')

---
## EXP 13: Statistical Significance

In [ ]:
# ── EXP 13: Paired t-test from 10-seed results ──
from scipy import stats

def compute_significance(results_list, methods_map=METHODS):
    """Compute paired t-tests: UNIFEWS vs FA-adapt on each dataset."""
    # Group by (dataset, method, seed)
    data = defaultdict(lambda: defaultdict(dict))
    for r in results_list:
        if not r.get('acc'): continue
        mname = 'unknown'
        for name, (algo, fa, ta, tw) in methods_map.items():
            if (r['algo'] == algo and abs(r['fa_alpha'] - fa) < 0.01
                and abs(r['thr_a'] - ta) < 0.01):
                mname = name; break
        data[r['config']][mname][r['seed']] = r['acc']
    
    print(f'{"Dataset":15s} | {"UNIFEWS":>12s} | {"FA-adapt":>12s} | {"Δ":>8s} | {"p-value":>10s} | Sig?')
    print('-' * 75)
    for ds in DS_ALL:
        if ds not in data: continue
        u_accs = sorted(data[ds].get('UNIFEWS', {}).items())
        f_accs = sorted(data[ds].get('FA-adapt', {}).items())
        if len(u_accs) < 2 or len(f_accs) < 2: continue
        
        # Align by seed
        common_seeds = set(dict(u_accs).keys()) & set(dict(f_accs).keys())
        if len(common_seeds) < 2: continue
        u_vals = [dict(u_accs)[s]*100 for s in sorted(common_seeds)]
        f_vals = [dict(f_accs)[s]*100 for s in sorted(common_seeds)]
        
        t_stat, p_val = stats.ttest_rel(f_vals, u_vals)
        delta = np.mean(f_vals) - np.mean(u_vals)
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
        
        print(f'{ds:15s} | {np.mean(u_vals):6.2f}±{np.std(u_vals):5.2f} | '
              f'{np.mean(f_vals):6.2f}±{np.std(f_vals):5.2f} | '
              f'{delta:+6.2f} | {p_val:10.4f} | {sig}')

# Use 10-seed if available, otherwise 1-seed
if 'res_main_10seed' in dir():
    compute_significance(res_main_10seed)
else:
    print('10-seed results not available. Using 1-seed (no significance test possible).')
    compute_significance(res_main_1seed)

---
## Generate LaTeX Tables

In [ ]:
# ── Generate LaTeX for main table (Table 3) ──
def generate_latex_main_table(results, output_path=None):
    """Generate LaTeX table from results."""
    # Build data structure
    data = defaultdict(lambda: defaultdict(list))
    for r in results:
        if not r.get('acc'): continue
        mname = 'unknown'
        for name, (algo, fa, ta, tw) in METHODS.items():
            if (r['algo'] == algo and abs(r['fa_alpha'] - fa) < 0.01
                and abs(r['thr_a'] - ta) < 0.01):
                mname = name; break
        data[mname][r['config']].append(r['acc'] * 100)
    
    datasets = DS_ALL
    methods = METHOD_ORDER
    
    # Find best per column
    best = {}
    for ds in datasets:
        vals = {}
        for m in methods:
            if data[m][ds]:
                vals[m] = np.mean(data[m][ds])
        if vals:
            best[ds] = max(vals, key=vals.get)
    
    # Generate LaTeX
    ncols = len(datasets)
    header = ' & '.join(['Method'] + [d.replace('_', '-').capitalize() for d in datasets])
    
    lines = []
    lines.append('\\begin{table}[t]')
    lines.append('\\centering')
    lines.append('\\caption{Main results: test accuracy (\\%) on 9 benchmarks. '
                 'Bold = best, underline = second best.}')
    lines.append('\\label{tab:main}')
    lines.append(f'\\begin{{tabular}}{{l{"c"*ncols}}}')
    lines.append('\\toprule')
    lines.append(header + ' \\\\')
    lines.append('\\midrule')
    
    for m in methods:
        cells = [m.replace('FA-', 'FA-UNIFEWS$_{\\text{').replace(
                 'static', 'static}}$').replace('adapt', 'adapt}}$')]
        for ds in datasets:
            vals = data[m][ds]
            if vals:
                mean = np.mean(vals)
                std = np.std(vals) if len(vals) > 1 else 0
                cell = f'{mean:.2f}'
                if std > 0: cell += f'$\\pm${std:.2f}'
                if best.get(ds) == m:
                    cell = f'\\textbf{{{cell}}}'
            else:
                cell = '--'
            cells.append(cell)
        lines.append(' & '.join(cells) + ' \\\\')
        if m == 'Random':  # Add \midrule after baselines
            lines.append('\\midrule')
    
    lines.append('\\bottomrule')
    lines.append('\\end{tabular}')
    lines.append('\\end{table}')
    
    latex = '\n'.join(lines)
    if output_path:
        with open(output_path, 'w') as f:
            f.write(latex)
        print(f'Saved to {output_path}')
    print(latex)
    return latex

# Generate from available results
results_for_table = res_main_10seed if 'res_main_10seed' in dir() else res_main_1seed
generate_latex_main_table(results_for_table,
                          str(RESULTS_DIR / 'tables' / 'table3_main.tex'))

---
## Aggregate & Save All Results

In [ ]:
# ── Combine all experiment results ──
def results_to_df(results, label=''):
    rows = []
    for r in results:
        if r is None: continue
        row = {k: r.get(k) for k in ['config','algo','seed','thr_a','thr_w',
                                      'fa_alpha','acc','numel_a','numel_w',
                                      'time_train','time_test','macs_test','macs_train']}
        row['experiment'] = label
        rows.append(row)
    return pd.DataFrame(rows)

all_dfs = []
for name, var_name in [
    ('main_1seed', 'res_main_1seed'),
    ('shock', 'res_shock'),
    ('backbone', 'res_backbone'),
    ('deep', 'res_deep'),
    ('decoupled', 'res_decoupled'),
    ('ablation', 'res_ablation'),
    ('sparsity', 'res_sparsity'),
    ('main_10seed', 'res_main_10seed'),
]:
    g = globals()
    if var_name in g and g[var_name]:
        df = results_to_df(g[var_name], label=name)
        if len(df) > 0:
            all_dfs.append(df)
            print(f'  {name}: {len(df)} rows')

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    output = RESULTS_DIR / 'all_experiments_v2.csv'
    df_all.to_csv(output, index=False)
    print(f'\nTotal: {len(df_all)} rows saved to {output}')
else:
    print('No results to save.')

In [ ]:
# ── Figure inventory ──
print('=' * 60)
print('FIGURE INVENTORY')
print('=' * 60)
for f in sorted((RESULTS_DIR / 'figures').glob('*.pdf')):
    size = f.stat().st_size / 1024
    print(f'  {f.name:40s} {size:6.1f} KB')

print(f'\nTotal figures: {len(list((RESULTS_DIR / "figures").glob("*.pdf")))}')

---
## Summary of Key Findings

After running all experiments, fill in the key numbers:

| Claim | Evidence |
|---|---|
| Computational Shock on heterophilic datasets | Fig 1: UNIFEWS < MLP on Chameleon/Cornell/Texas/Wisconsin |
| FA-UNIFEWS rescues from shock | Fig 1: FA-adapt stays above MLP line |
| +X% on Chameleon | Table 3: UNIFEWS → FA-adapt delta |
| Edge homophily purification | Fig 2: h increases after FA-UNIFEWS pruning |
| <3% MACs overhead | Fig 9: UNIFEWS vs FA-adapt MACs difference |
| Backbone transferable | Fig 3: Works with GCN/GAT/SAGE |
| Depth robust | Fig 4: FA-adapt maintains accuracy at 16 layers |
| Adaptive > Static on larger graphs | Table 3 + Fig 5 |
| Interpretable gate | Fig 7: Cora high α, Chameleon low α |
| Faster convergence | Fig 8: FA-adapt converges earlier |
| Better representations | Fig 10: Tighter t-SNE clusters |